# Spatial Features for DTP

Пространственное обогащение ДТП: H3-индекс, OSMnx-признаки, парсинг текстовых полей.

In [ ]:
import pandas as pd
import numpy as np
import re, ast, os, time, json, warnings
warnings.filterwarnings("ignore")

import h3
import osmnx as ox
from shapely.geometry import Point
import geopandas as gpd
from scipy.spatial import cKDTree

ox.settings.log_console = False
ox.settings.use_cache = True
ox.settings.cache_folder = ".osmnx_cache"

print("Библиотеки загружены.")
print(f"h3=={h3.__version__}, osmnx=={ox.__version__}")

In [ ]:
df = pd.read_csv("output/fact_dtp.csv")
df["moment_date"] = pd.to_datetime(df["moment_date"], errors="coerce")

# Фильтр: только 2022-2024
df = df[df["moment_date"].dt.year.isin([2022, 2023, 2024])].reset_index(drop=True)
print(f"Строк после фильтра по годам: {len(df)}")

## 1. Проверка и очистка координат

In [ ]:
# Москва: bbox lat 55.1-56.1, lon 36.8-38.0
LAT_MIN, LAT_MAX = 55.1, 56.1
LON_MIN, LON_MAX = 36.8, 38.0

total = len(df)
null_mask = df["place_latitude"].isna() | df["place_longitude"].isna()
zero_mask = (df["place_latitude"] == 0) | (df["place_longitude"] == 0)
bbox_mask = ~(
    df["place_latitude"].between(LAT_MIN, LAT_MAX) &
    df["place_longitude"].between(LON_MIN, LON_MAX)
)
invalid_mask = null_mask | zero_mask | bbox_mask

print(f"Всего ДТП:               {total}")
print(f"Пропуски координат:      {null_mask.sum()}")
print(f"Нулевые координаты:      {zero_mask.sum()}")
print(f"Вне bbox Москвы/МО:      {bbox_mask.sum()}")
print(f"Итого невалидных:        {invalid_mask.sum()} ({invalid_mask.sum()/total*100:.1f}%)")
print(f"Валидных для обработки:  {(~invalid_mask).sum()} ({(~invalid_mask).sum()/total*100:.1f}%)")

df["coord_valid"] = (~invalid_mask).astype(int)
df_valid = df[~invalid_mask].copy().reset_index(drop=True)
print(f"\nРабочий датасет: {len(df_valid)} строк")

## 2. H3-индекс

In [ ]:
# res=8 -> ~0.74 km2 (квартал), res=9 -> ~0.11 km2 (перекрёсток)
H3_RES_MAIN   = 8
H3_RES_DETAIL = 9

def to_h3(lat, lon, res):
    try:
        return h3.latlng_to_cell(lat, lon, res)
    except Exception:
        return None

df_valid["h3_r8"] = df_valid.apply(
    lambda r: to_h3(r["place_latitude"], r["place_longitude"], H3_RES_MAIN), axis=1)
df_valid["h3_r9"] = df_valid.apply(
    lambda r: to_h3(r["place_latitude"], r["place_longitude"], H3_RES_DETAIL), axis=1)

print(f"Уникальных ячеек res=8:  {df_valid["h3_r8"].nunique()}")
print(f"Уникальных ячеек res=9:  {df_valid["h3_r9"].nunique()}")
print(df_valid[["dtp_id","place_latitude","place_longitude","h3_r8","h3_r9"]].head(3))

## 3. Парсинг текстовых полей (объекты на месте ДТП)

Поля `road_constructions_here` и `road_constructions_there` содержат Python-списки в виде строк — извлекаем бинарные признаки без запросов к API.

In [ ]:
def parse_list_field(val):
    """Парсит строку вида ['A', 'B'] в список Python."""
    if pd.isna(val) or val == "":
        return []
    try:
        result = ast.literal_eval(val)
        if isinstance(result, list):
            return [str(x).strip() for x in result]
    except Exception:
        pass
    cleaned = re.sub(r"[\[\]'\""]", "", str(val))
    return [x.strip() for x in cleaned.split(",") if x.strip()]

df_valid["here_list"]  = df_valid["road_constructions_here"].apply(parse_list_field)
df_valid["there_list"] = df_valid["road_constructions_there"].apply(parse_list_field)
df_valid["all_objects"] = df_valid["here_list"] + df_valid["there_list"]

def has_kw(lst, *keywords):
    text = " ".join(lst).lower()
    return int(any(kw in text for kw in keywords))

df_valid["has_traffic_light"]     = df_valid["all_objects"].apply(has_kw, args=("регулируемый перекресток","регулируемый перекрёсток","регулируемый пешеходный",))
df_valid["has_crosswalk"]         = df_valid["all_objects"].apply(has_kw, args=("пешеходный переход",))
df_valid["has_unregulated_cross"] = df_valid["all_objects"].apply(has_kw, args=("нерегулируемый пешеходный",))
df_valid["has_regulated_cross"]   = df_valid["all_objects"].apply(has_kw, args=("регулируемый пешеходный",))
df_valid["has_bus_stop"]          = df_valid["all_objects"].apply(has_kw, args=("остановка",))
df_valid["has_roundabout"]        = df_valid["all_objects"].apply(has_kw, args=("кольцо","круговое",))
df_valid["has_bridge"]            = df_valid["all_objects"].apply(has_kw, args=("мост","эстакада","путепровод",))
df_valid["has_underground_cross"] = df_valid["all_objects"].apply(has_kw, args=("подземный пешеходный",))
df_valid["has_school_nearby"]     = df_valid["all_objects"].apply(has_kw, args=("школ","образовательн","детск",))
df_valid["has_residential"]       = df_valid["all_objects"].apply(has_kw, args=("жилые","многоквартирные",))
df_valid["has_gas_station"]       = df_valid["all_objects"].apply(has_kw, args=("азс",))
df_valid["has_camera"]            = df_valid["all_objects"].apply(has_kw, args=("камер","фотовидеофиксаци",))
df_valid["is_intersection"]       = df_valid["all_objects"].apply(has_kw, args=("перекресток","перекрёсток",))
df_valid["is_open_road"]          = df_valid["here_list"].apply(has_kw, args=("перегон",))

df_valid["poi_count_from_data"] = df_valid["there_list"].apply(
    lambda lst: 0 if (len(lst)==0 or any("отсутстви" in x.lower() for x in lst)) else len(lst))

print("Бинарные признаки рассчитаны.")
cols = ["has_traffic_light","has_crosswalk","has_bus_stop","is_intersection","has_camera","poi_count_from_data"]
print(df_valid[cols].mean().round(3))

## 4. Признаки геометрии дороги (из исходных данных)

In [ ]:
df_valid["lane_width_m"] = np.where(
    df_valid["traffic_lane_amount"] > 0,
    df_valid["traffic_area_width"] / df_valid["traffic_lane_amount"],
    np.nan
)
df_valid["has_sidewalk"]        = (df_valid["sidewalk_width"] > 0).astype(int)
df_valid["has_center_divider"]  = (df_valid["center_mall_width"] > 0).astype(int)
df_valid["is_multilane"]        = (df_valid["traffic_lane_amount"] >= 4).astype(int)
df_valid["dtp_lane_normalized"] = np.where(
    df_valid["traffic_lane_amount"] > 0,
    df_valid["dtp_traffic_lane"] / df_valid["traffic_lane_amount"],
    np.nan
)

for col in ["srf_code","tr_area_state_code","cmall_type_code","cut_code","cut_pr_code"]:
    if col in df_valid.columns:
        df_valid[col] = df_valid[col].astype("category")

print("Геометрические признаки:")
print(df_valid[["lane_width_m","has_sidewalk","has_center_divider","is_multilane"]].describe())

## 5. OSMnx: расстояния и плотности объектов

> Запросы батчами по уникальным H3-ячейкам (res=8). Каждая ячейка обрабатывается один раз — результат кэшируется на диск в `output_spatial/osm_cell_cache.json`. При прерывании следующий запуск продолжит с незакэшированных ячеек.

In [ ]:
DIST_QUERY = 500  # метры — радиус запроса OSM вокруг центра ячейки
RADII = [100, 300]  # радиусы для подсчёта плотности

OSM_TAGS = {
    "traffic_signals": {"highway": "traffic_signals"},
    "crosswalk":       {"highway": "crossing"},
    "bus_stop":        {"highway": "bus_stop"},
}

_EMPTY = {
    "dist_to_signal_m": None, "dist_to_cross_m": None, "dist_to_bus_m": None,
    "signal_cnt_100m": None, "signal_cnt_300m": None,
    "cross_cnt_100m":  None, "cross_cnt_300m":  None,
    "bus_cnt_100m":    None, "bus_cnt_300m":    None,
    "speed_limit": None, "road_class": None, "landuse_type": None,
}

def osm_features_for_cell(cell_lat, cell_lon):
    result = _EMPTY.copy()
    try:
        G = ox.graph_from_point((cell_lat, cell_lon), dist=DIST_QUERY,
                                network_type="drive", simplify=True)
        edges = ox.graph_to_gdfs(G, nodes=False)
        edges_proj = edges.to_crs("EPSG:32637")
        pt_proj = gpd.GeoSeries(
            [Point(cell_lon, cell_lat)], crs="EPSG:4326"
        ).to_crs("EPSG:32637").iloc[0]
        edges_proj["_d"] = edges_proj.geometry.distance(pt_proj)
        nearest = edges_proj.loc[edges_proj["_d"].idxmin()]
        speed_raw = nearest.get("speed_kph", None)
        if speed_raw is not None:
            result["speed_limit"] = float(speed_raw[0]) if isinstance(speed_raw, list) else float(speed_raw)
        hw = nearest.get("highway", None)
        result["road_class"] = hw[0] if isinstance(hw, list) else hw

        tags_pts = {}
        for name, tags in OSM_TAGS.items():
            try:
                gdf = ox.features_from_point((cell_lat, cell_lon), tags=tags, dist=DIST_QUERY)
                gdf = gdf[gdf.geometry.type == "Point"].to_crs("EPSG:32637")
                tags_pts[name] = np.array([(g.x, g.y) for g in gdf.geometry])
            except Exception:
                tags_pts[name] = np.empty((0, 2))

        pt_xy = np.array([[pt_proj.x, pt_proj.y]])

        def nearest_dist(coords):
            if len(coords) == 0: return None
            d, _ = cKDTree(coords).query(pt_xy)
            return float(d[0])

        def count_within(coords, r):
            if len(coords) == 0: return 0
            return int(len(cKDTree(coords).query_ball_point(pt_xy[0], r)))

        result["dist_to_signal_m"] = nearest_dist(tags_pts["traffic_signals"])
        result["dist_to_cross_m"]  = nearest_dist(tags_pts["crosswalk"])
        result["dist_to_bus_m"]    = nearest_dist(tags_pts["bus_stop"])
        for r in RADII:
            result[f"signal_cnt_{r}m"] = count_within(tags_pts["traffic_signals"], r)
            result[f"cross_cnt_{r}m"]  = count_within(tags_pts["crosswalk"], r)
            result[f"bus_cnt_{r}m"]    = count_within(tags_pts["bus_stop"], r)

        try:
            lu = ox.features_from_point((cell_lat, cell_lon), tags={"landuse": True}, dist=200)
            if len(lu) > 0:
                lu_proj = lu.to_crs("EPSG:32637")
                lu_proj["_d"] = lu_proj.geometry.distance(pt_proj)
                result["landuse_type"] = str(lu_proj.loc[lu_proj["_d"].idxmin(), "landuse"])
        except Exception:
            pass

    except Exception:
        pass
    return result

print("Функция osm_features_for_cell определена.")
print(f"Уникальных H3 res=8 ячеек: {df_valid["h3_r8"].nunique()}")

In [ ]:
CACHE_FILE = "output_spatial/osm_cell_cache.json"
os.makedirs("output_spatial", exist_ok=True)

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        cell_cache = json.load(f)
    print(f"Загружен кэш: {len(cell_cache)} ячеек")
else:
    cell_cache = {}
    print("Кэш пуст, начинаем с нуля")

unique_cells = (
    df_valid.dropna(subset=["h3_r8"])
    .drop_duplicates("h3_r8")[["h3_r8"]]
    .copy()
)
unique_cells["cell_lat"] = unique_cells["h3_r8"].apply(lambda c: h3.cell_to_latlng(c)[0])
unique_cells["cell_lon"] = unique_cells["h3_r8"].apply(lambda c: h3.cell_to_latlng(c)[1])

total_cells = len(unique_cells)
processed = 0
print(f"Нужно обработать: {total_cells} | в кэше: {len(cell_cache)} | осталось: {total_cells - len(cell_cache)}")

try:
    for _, row in unique_cells.iterrows():
        cell_id = row["h3_r8"]
        if cell_id in cell_cache:
            processed += 1
            continue

        feats = osm_features_for_cell(row["cell_lat"], row["cell_lon"])
        cell_cache[cell_id] = feats
        processed += 1

        if processed % 50 == 0:
            with open(CACHE_FILE, "w", encoding="utf-8") as f:
                json.dump(cell_cache, f, ensure_ascii=False)
            print(f"  [{processed}/{total_cells}] сохранён кэш ({len(cell_cache)} ячеек)")

        time.sleep(0.05)

except KeyboardInterrupt:
    print(f"\nПрервано на {processed}/{total_cells}. Сохраняем кэш...")

# Финальное сохранение
with open(CACHE_FILE, "w", encoding="utf-8") as f:
    json.dump(cell_cache, f, ensure_ascii=False)
print(f"\n✅ OSM готово. Кэш: {len(cell_cache)} ячеек.")

## 6. Сборка feat_spatial_dtp

In [ ]:
# Джойним OSM из кэша через h3_r8
osm_df = pd.DataFrame.from_dict(cell_cache, orient="index")
osm_df.index.name = "h3_r8"
osm_df = osm_df.reset_index()

df_valid = df_valid.merge(osm_df, on="h3_r8", how="left")

SPATIAL_COLS = [
    "dtp_id", "h3_r8", "h3_r9", "coord_valid",
    # из текстовых полей
    "has_traffic_light","has_crosswalk","has_unregulated_cross","has_regulated_cross",
    "has_bus_stop","has_roundabout","has_bridge","has_underground_cross",
    "has_school_nearby","has_residential","has_gas_station","has_camera",
    "is_intersection","is_open_road","poi_count_from_data",
    # геометрия дороги
    "traffic_lane_amount","dtp_traffic_lane","traffic_area_width",
    "wayside_width","sidewalk_width","center_mall_width",
    "lane_width_m","has_sidewalk","has_center_divider","is_multilane","dtp_lane_normalized",
    "srf_code","tr_area_state_code","cmall_type_code","cut_code","cut_pr_code",
    # OSMnx
    "dist_to_signal_m","dist_to_cross_m","dist_to_bus_m",
    "signal_cnt_100m","signal_cnt_300m",
    "cross_cnt_100m","cross_cnt_300m",
    "bus_cnt_100m","bus_cnt_300m",
    "speed_limit","road_class","landuse_type",
]

feat_spatial_dtp = df_valid[[c for c in SPATIAL_COLS if c in df_valid.columns]].copy()
feat_spatial_dtp.to_csv("output_spatial/feat_spatial_dtp.csv", index=False)
print(f"✅ feat_spatial_dtp.csv: {feat_spatial_dtp.shape}")

## 7. Агрегат agg_cell_space (по H3 ячейкам)

In [ ]:
agg_dict = {
    "dtp_id": "count",
    "has_traffic_light": "mean", "has_crosswalk": "mean",
    "has_bus_stop": "mean", "is_intersection": "mean",
    "is_open_road": "mean", "has_camera": "mean",
    "poi_count_from_data": "mean", "traffic_lane_amount": "mean",
    "lane_width_m": "mean", "is_multilane": "mean",
    "dist_to_signal_m": "mean", "dist_to_cross_m": "mean", "dist_to_bus_m": "mean",
    "signal_cnt_300m": "mean", "cross_cnt_300m": "mean", "bus_cnt_300m": "mean",
    "speed_limit": "mean",
}
agg_dict = {k: v for k, v in agg_dict.items() if k in feat_spatial_dtp.columns}

agg_cell_space = feat_spatial_dtp.groupby("h3_r8").agg(agg_dict).reset_index()
agg_cell_space.rename(columns={"dtp_id": "dtp_count"}, inplace=True)

for col in ["road_class", "landuse_type"]:
    if col in feat_spatial_dtp.columns:
        mode_s = feat_spatial_dtp.groupby("h3_r8")[col].agg(
            lambda x: x.mode().iloc[0] if not x.mode().empty else None
        )
        agg_cell_space = agg_cell_space.merge(mode_s.rename(col), on="h3_r8", how="left")

agg_cell_space.to_csv("output_spatial/agg_cell_space.csv", index=False)
print(f"✅ agg_cell_space.csv: {agg_cell_space.shape}")
print(agg_cell_space.head(3).to_string())

## 8. Карточка качества геоданных

In [ ]:
total_orig  = len(df)
total_valid = len(df_valid)
total_with_h3  = df_valid["h3_r8"].notna().sum()
total_osm = feat_spatial_dtp["dist_to_signal_m"].notna().sum() if "dist_to_signal_m" in feat_spatial_dtp.columns else 0

lines = [
    "КАРТОЧКА КАЧЕСТВА ГЕОДАННЫХ",
    "=" * 40,
    f"Входных ДТП (2022-2024):      {total_orig}",
    f"Валидных координат:           {total_valid}  ({total_valid/total_orig*100:.1f}%)",
    f"Получили H3-индекс:           {total_with_h3}  ({total_with_h3/total_orig*100:.1f}%)",
    f"Обогащены OSM-данными:        {total_osm}  ({total_osm/max(total_valid,1)*100:.1f}% от валидных)",
    "",
    "ПОКРЫТИЕ ТЕКСТОВЫХ ПРИЗНАКОВ (% ДТП)",
    "-" * 40,
]
for col in ["has_traffic_light","has_crosswalk","has_bus_stop","is_intersection",
            "has_camera","has_bridge","has_school_nearby","is_open_road"]:
    if col in feat_spatial_dtp.columns:
        pct = feat_spatial_dtp[col].mean() * 100
        lines.append(f"  {col:<35} {pct:.1f}%")

lines += ["", "ПОКРЫТИЕ OSM-ПРИЗНАКОВ", "-" * 40]
for col in ["dist_to_signal_m","dist_to_cross_m","dist_to_bus_m","speed_limit","road_class","landuse_type"]:
    if col in feat_spatial_dtp.columns:
        pct = feat_spatial_dtp[col].notna().mean() * 100
        lines.append(f"  {col:<35} {pct:.1f}%")

report = "\n".join(lines)
print(report)
os.makedirs("output_spatial", exist_ok=True)
with open("output_spatial/coverage_report.txt", "w", encoding="utf-8") as f:
    f.write(report)
print("\n✅ coverage_report.txt сохранён")

In [ ]:
readme = """
# spatial_features — README

## Источники данных

| Признак | Источник | Логика |
|---|---|---|
| h3_r8, h3_r9 | Uber H3 | latlng_to_cell(lat, lon, res=8/9) |
| has_traffic_light, has_crosswalk, ... | road_constructions_here / there | Парсинг Python-списков, поиск ключевых слов |
| poi_count_from_data | road_constructions_there | Длина списка объектов притяжения (без «Отсутствие...») |
| lane_width_m | traffic_area_width / traffic_lane_amount | Средняя ширина полосы, метры |
| dist_to_signal_m | OSMnx + highway=traffic_signals | cKDTree — до ближайшего светофора, м |
| dist_to_cross_m | OSMnx + highway=crossing | cKDTree — до ближайшего перехода, м |
| dist_to_bus_m | OSMnx + highway=bus_stop | cKDTree — до ближайшей остановки, м |
| signal/cross/bus_cnt_Xm | OSMnx | Количество объектов в радиусе X метров |
| speed_limit | OSMnx edges (maxspeed/speed_kph) | Ближайший дорожный сегмент, км/ч |
| road_class | OSMnx edges (highway tag) | Тип дороги: motorway/primary/residential/... |
| landuse_type | OSMnx landuse | Ближайший landuse-полигон в 200 м |

## Единицы измерения

- Расстояния: **метры**
- Плотности: **количество объектов** в радиусе 100 м / 300 м
- Ширины полос и дороги: **метры**
- Бинарные флаги: **0 / 1**
- Скорость: **км/ч**

## Воспроизводимость

1. Ячейки 1–5 (загрузка, QA, H3, парсинг, геометрия) — не требуют сети
2. OSM-цикл — кэш пишется в `output_spatial/osm_cell_cache.json` каждые 50 ячеек
3. При прерывании повторный запуск продолжит с незакэшированных ячеек
4. После заполнения кэша — ячейки сборки и агрегации

## Выходные файлы

- `output_spatial/feat_spatial_dtp.csv` — ключ `dtp_id`
- `output_spatial/agg_cell_space.csv` — ключ `h3_r8`
- `output_spatial/coverage_report.txt` — отчёт по покрытию
- `output_spatial/osm_cell_cache.json` — кэш OSM по ячейкам (res=8)
"""

with open("output_spatial/README.md", "w", encoding="utf-8") as f:
    f.write(readme)
print("✅ README.md сохранён")
print(readme)